# Agents in LangChain

- LLM is more of a reasoning engine where you can give chunks of text or other sources of information. 
- LangChain's agents framework helps to answer questions, reason through content or decide next steps.
- Agents are powerful tools
- LLM Model is the reasoning engine of an agent 

#### _Agent that can answer Math and Search questions:_

Import these from langchain:

    from langchain.agents.agent_toolkits import create_python_agent
    from langchain.agents import load_tools, initialize_agent
    from langchain.agents import AgentType
    from langchain.tools.python.tool import PythonREPLTool
    from langchain.python import PythonREPL
    from langchain.chat_models import ChatOpenAI

Initialise LLM Model:

    llm = ChatOpenAI(temperature=0, model=llm_model)

Load tools needed for the agent:

    # llm-math tool is also a langchain which uses LLM along with a calculator to solve math problems
    # wikipedia tool is an api that connects to wikipedia to run search queries

    tools = load_tools(["llm-math","wikipedia"], llm=llm)

Intialise Agent with tools, llm model, agent type:

- Chat zero shot react desciption is used as agent type to get best reasoning results from a model
- Handle parsing errors is set to true
- Verbose and debug is set to true to understand the steps involved for the agent

    agent= initialize_agent(
        tools, 
        llm, 
        agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
        handle_parsing_errors=True,
        verbose = True)

_EXAMPLE: Ask agent a Math question_

    agent("What is the 25% of 300?")

=> Agent finds action and action input needed for the question. 

=> In this example, action is to open calculator and action input is to put the math equation in the calculator. Observation result is coming from the calculator tool. 

_EXAMPLE: Ask agent a broad search question_

    question = "Tom M. Mitchell is an American computer scientist \
    and the Founders University Professor at Carnegie Mellon University (CMU)\
    what book did he write?"
    
    result = agent(question) 

=> Agent understands that the input is a search question and action is to open wikipedia tool. 

=> Observation send data about the question from wikipedia and finds the specific answer for the question based on text similarity.

#### _Create a Python Agent:_

- Can be used to answer code questions. 
- LLM models have a code interpretor which is used to write code and execute code. 

Create Python agent with llm model, python repl tool:

(REPL is a way to interact with code like jupyter notbook so that model can execute code)

    agent = create_python_agent(
        llm,
        tool=PythonREPLTool(),
        verbose=True
    )

_EXAMPLE: Ask agent to sort a list of names by last names and then first names_

    customer_list = [["Harrison", "Chase"], 
                    ["Lang", "Chain"],
                    ["Dolly", "Too"],
                    ["Elle", "Elem"], 
                    ["Geoff","Fusion"], 
                    ["Trance","Former"],
                    ["Jen","Ayai"]
                    ]

    agent.run(f"""Sort these customers by \
    last name and then first name \
    and print the output: {customer_list}""") 

=> In this example, model realises it can use sorted function to sort names. Sorts them by last names and then by first names.

=> Uses python repl tool as action and the code function is the action input. 

=> Observation is what is result after action is exceuted on action input.

LangChain debug:

Can understand how the agent works in more detail by setting LangChain debug to true. There is a stop method for the model to know when to end generations.

    import langchain
    langchain.debug=True
    agent.run(f"""Sort these customers by \
    last name and then first name \
    and print the output: {customer_list}""") 
    langchain.debug=False

#### _Create a Custom Agent and Tool:_

- You can connect agents to your own sources of information, tools, api or data. 
- You can create custom tools.

_EXAMPLE: Tool to tell the current date_

Import tool decorator which can be applied to any function to turn into a tool that LangChain can use:

    from langchain.agents import tool
    from datetime import date

Define time function in the tool:

    @tool
    def time(text: str) -> str:
        """Returns todays date, use this for any \
        questions related to knowing todays date. \
        The input should always be an empty string, \
        and this function will always return todays \
        date - any date mathmatics should occur \
        outside this function."""
        return str(date.today())

Initialse agent and add time tool into the existing tools:

    agent= initialize_agent(
        tools + [time], 
        llm, 
        agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
        handle_parsing_errors=True,
        verbose = True)

Call agent:

    agent("whats the date today?")

=> It recognises it should use the custom tool as the action. 

=> The action input is an empty string as specified in the function. 

=> Returns observation from action tool and the llm returns the observation.